# Dense Mapped Logit-KD: INT8 Validation Preflight

Notebook này chỉ bắt đầu sau khi package ONNX FP32 seed 123 đã vượt ONNX checker
và parity PyTorch–ONNX Runtime. Mục tiêu là tạo và sàng lọc ứng viên INT8 mà
không huấn luyện lại, không chọn lại checkpoint và không mở Held-out Test.

## Thiết kế khoa học

1. xác minh toàn vẹn package FP32 và authorization INT8;
2. khóa **128 ảnh Karpathy Validation** bằng SHA-256 trước khi quantization;
3. dùng **32 ảnh Validation holdout**, không giao với calibration, để đo parity;
4. thử `static QDQ INT8` và `dynamic weight INT8`;
5. chỉ chọn ứng viên nhỏ nhất vượt cosine, top-1, finite/reload và size gate;
6. chỉ khi preflight đạt mới authorize full 5.000-image Validation quality gate.

## Kaggle Input bắt buộc — đúng 4 input

1. output `XLKD05_Dense_Mapped_LogitKD_production_onnx_fp32`;
2. output notebook `01_MSCOCO_Data_Contract`;
3. dataset `mscoco2014-1`;
4. dataset `ONNX_Runtime_Python313_Offline`.

Không gắn output Held-out Test riêng, seed 42/2026, Teacher, training checkpoint
hoặc `pycocoevalcap` ở bước này. Runtime: **CPU / Accelerator None**, Internet Off.


## Cell 1 — Authorization, integrity và Validation-only boundary


In [1]:
import csv
import hashlib
import importlib
import importlib.util
import json
import math
import os
import platform
import shutil
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
assert INPUT_ROOT.is_dir(), "Notebook phải chạy trên Kaggle."

RUN_NAME = "XLKD05_Dense_Mapped_LogitKD_int8_validation_preflight"
OUT = WORK_ROOT / RUN_NAME
AUDITS = OUT / "audits"
CONFIGS = OUT / "configs"
CANDIDATES = OUT / "candidates"
PACKAGE = OUT / "package"
RESULTS = OUT / "results"
MANIFESTS = OUT / "manifests"
for directory in (OUT, AUDITS, CONFIGS, CANDIDATES, PACKAGE, RESULTS, MANIFESTS):
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)

def exactly_one(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, f"Cần đúng 1 file {name!r}; tìm thấy {len(matches)}: {matches[:10]}"
    return matches[0]

def verify_manifest(root, manifest_path, expected_rows):
    with Path(manifest_path).open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(expected_rows), (len(rows), expected_rows)
    for row in rows:
        target = Path(root) / row["relative_path"]
        assert target.is_file(), target
        assert target.stat().st_size == int(row["bytes"]), target
        assert sha256(target) == row["sha256"], target
    return rows

# Kaggle Python 3.13 có thể thiếu ORT. Chỉ cài wheel offline đã gắn.
if importlib.util.find_spec("onnxruntime") is None:
    ort_wheels = sorted(INPUT_ROOT.rglob("onnxruntime-*-cp313-cp313-manylinux*.whl"))
    assert len(ort_wheels) == 1, ort_wheels
    wheel_dir = ort_wheels[0].parent
    support = []
    for pattern in ("flatbuffers-*.whl", "packaging-*.whl", "protobuf-*.whl"):
        found = sorted(wheel_dir.glob(pattern))
        assert len(found) == 1, (pattern, found)
        support.append(found[0])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
        *[str(path) for path in support], str(ort_wheels[0]),
    ])
    importlib.invalidate_caches()

required_modules = ("numpy", "PIL", "torch", "torchvision", "onnx", "onnxruntime")
runtime_modules = {name: importlib.util.find_spec(name) is not None for name in required_modules}
assert all(runtime_modules.values()), runtime_modules

COMPLETE_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_PRODUCTION_ONNX_FP32_COMPLETE.json"
)
AUTH_PATH = exactly_one("INT8_VALIDATION_GATE_AUTHORIZED.json")
complete = read_json(COMPLETE_PATH)
authorization = read_json(AUTH_PATH)
assert complete["status"] == "complete"
assert complete["decision"] == "fp32_onnx_parity_passed_int8_validation_gate_authorized"
assert int(complete["selected_seed"]) == 123
assert complete["fp32_parity"] == "passed"
assert complete["new_test_opening"] is False
assert authorization["status"] == "authorized"
assert authorization["constraints"]["calibration_split"] == "validation_only"
assert authorization["constraints"]["test_opening"] is False

FP32_ROOT = COMPLETE_PATH.parent
manifest_reference = complete["manifest"]
FP32_MANIFEST = FP32_ROOT / manifest_reference["relative_path"]
assert sha256(FP32_MANIFEST) == manifest_reference["sha256"]
fp32_manifest_rows = verify_manifest(FP32_ROOT, FP32_MANIFEST, manifest_reference["rows"])

DEPLOYMENT_CONTRACT = FP32_ROOT / complete["package_contract"]["relative_path"]
assert sha256(DEPLOYMENT_CONTRACT) == complete["package_contract"]["sha256"]
deployment = read_json(DEPLOYMENT_CONTRACT)
FP32_ENCODER = DEPLOYMENT_CONTRACT.parent / deployment["encoder"]["file"]
FP32_DECODER = DEPLOYMENT_CONTRACT.parent / deployment["decoder"]["file"]
VOCAB_PATH = DEPLOYMENT_CONTRACT.parent / deployment["artifacts"]["vocabulary"]["file"]
DECODER_CONTRACT_PATH = DEPLOYMENT_CONTRACT.parent / deployment["artifacts"]["decoder_contract"]["file"]
for path in (FP32_ENCODER, FP32_DECODER, VOCAB_PATH, DECODER_CONTRACT_PATH):
    assert path.is_file(), path

SPLIT_MANIFEST = exactly_one("karpathy_image_split_manifest.csv")
VALIDATION_CAPTIONS = exactly_one("karpathy_validation_captions_encoded.jsonl")
phase1_ready = read_json(exactly_one("PHASE1_MSCOCO_DATA_CONTRACT_READY.json"))
assert phase1_ready["status"] == "ready"
assert phase1_ready["test_set_status"] == "sealed"

preflight = {
    "schema_version": "dense_mapped_logit_kd_int8_preflight_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "selected_seed": 123,
    "fp32_manifest_rows_verified": len(fp32_manifest_rows),
    "fp32_encoder_sha256": sha256(FP32_ENCODER),
    "fp32_decoder_sha256": sha256(FP32_DECODER),
    "runtime_modules": runtime_modules,
    "environment": {"python": sys.version, "platform": platform.platform()},
    "new_training": False,
    "checkpoint_reselection": False,
    "test_data_read": False,
    "test_opening": False,
}
write_json(AUDITS / "cell01_entry_integrity.json", preflight)

print("INT8 authorization       : VERIFIED")
print("FP32 manifest            :", len(fp32_manifest_rows), "/", len(fp32_manifest_rows), "VERIFIED")
print("Selected seed            : 123")
print("Calibration split        : VALIDATION ONLY")
print("Held-out Test            : SEALED / NOT READ")


Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/flatbuffers-25.12.19-py2.py3-none-any.whl
Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/packaging-26.3-py3-none-any.whl
Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/protobuf-7.36.2-py3-none-any.whl
Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/onnxruntime-1.28.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
flatbuffers is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
packaging is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
INT8 authorization       : VERIFIED
FP32 manifest            : 

## Cell 2 — Khóa calibration cohort và disjoint parity cohort


In [2]:
CALIBRATION_COUNT = 128
PARITY_COUNT = 32
SELECTION_SALT = "DENSE_MAPPED_LOGITKD_INT8_VALIDATION_V1"

validation_rows = []
with SPLIT_MANIFEST.open("r", encoding="utf-8", newline="") as handle:
    for row in csv.DictReader(handle):
        if row["experiment_split"] == "validation":
            validation_rows.append({
                "image_id": int(row["image_id"]),
                "source_partition": row["source_partition"],
                "filename": Path(row["image_path"]).name,
            })
assert len(validation_rows) == 5000, len(validation_rows)
assert len({row["image_id"] for row in validation_rows}) == 5000

def rank_key(row):
    payload = f"{SELECTION_SALT}|{row['image_id']}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

ranked = sorted(validation_rows, key=lambda row: (rank_key(row), row["image_id"]))
calibration_rows = ranked[:CALIBRATION_COUNT]
parity_rows = ranked[CALIBRATION_COUNT:CALIBRATION_COUNT + PARITY_COUNT]
calibration_ids = {row["image_id"] for row in calibration_rows}
parity_ids = {row["image_id"] for row in parity_rows}
assert calibration_ids.isdisjoint(parity_ids)

assert all(row["source_partition"] == "val2014" for row in validation_rows)
root_matches = sorted(path.parent for path in INPUT_ROOT.rglob(calibration_rows[0]["filename"]) if path.is_file())
assert len(root_matches) == 1, root_matches
VALIDATION_IMAGE_ROOT = root_matches[0]
all_image_matches = {}
for row in calibration_rows + parity_rows:
    path = VALIDATION_IMAGE_ROOT / row["filename"]
    assert path.is_file(), path
    all_image_matches[row["image_id"]] = path

cohort_csv = CONFIGS / "int8_validation_cohorts.csv"
with cohort_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["cohort", "ordinal", "image_id", "source_partition", "filename", "selection_rank_sha256"])
    writer.writeheader()
    for cohort, rows in (("calibration", calibration_rows), ("parity_holdout", parity_rows)):
        for ordinal, row in enumerate(rows):
            writer.writerow({
                "cohort": cohort,
                "ordinal": ordinal,
                **row,
                "selection_rank_sha256": rank_key(row),
            })

cohort_contract = {
    "schema_version": "dense_mapped_logit_kd_int8_validation_cohorts_v1",
    "created_at_utc": now(),
    "status": "frozen_before_quantization",
    "selection_rule": "ascending SHA-256(salt|image_id)",
    "selection_salt": SELECTION_SALT,
    "validation_population": 5000,
    "calibration_count": CALIBRATION_COUNT,
    "parity_holdout_count": PARITY_COUNT,
    "disjoint": calibration_ids.isdisjoint(parity_ids),
    "calibration_image_ids": [row["image_id"] for row in calibration_rows],
    "parity_holdout_image_ids": [row["image_id"] for row in parity_rows],
    "cohort_csv_sha256": sha256(cohort_csv),
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(CONFIGS / "int8_validation_cohorts.json", cohort_contract)
write_json(AUDITS / "cell02_cohort_lock.json", {
    "status": "passed", **cohort_contract,
})

print("Validation population    : 5,000")
print("Calibration cohort       :", CALIBRATION_COUNT, "FROZEN")
print("Parity holdout cohort    :", PARITY_COUNT, "FROZEN")
print("Cohorts disjoint         : True")
print("Held-out Test            : SEALED / NOT USED")


Validation population    : 5,000
Calibration cohort       : 128 FROZEN
Parity holdout cohort    : 32 FROZEN
Cohorts disjoint         : True
Held-out Test            : SEALED / NOT USED


## Cell 3 — Preprocessing và CalibrationDataReader


In [3]:
import numpy as np
from PIL import Image
import torch
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import onnx
import onnxruntime as ort
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_dynamic,
    quantize_static,
)

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

VALIDATION_TRANSFORM = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

def preprocess_image(path):
    with Image.open(path) as handle:
        tensor = VALIDATION_TRANSFORM(handle.convert("RGB")).unsqueeze(0)
    return tensor.numpy().astype(np.float32, copy=False)

caption_rows = {}
with VALIDATION_CAPTIONS.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        image_id = int(row["image_id"])
        if image_id in calibration_ids or image_id in parity_ids:
            caption_rows.setdefault(image_id, []).append([int(token) for token in row["token_ids"]])
for image_id in calibration_ids | parity_ids:
    assert len(caption_rows.get(image_id, [])) >= 3, image_id

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1
session_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
PROVIDERS = ["CPUExecutionProvider"]
FP32_ENCODER_SESSION = ort.InferenceSession(str(FP32_ENCODER), sess_options=session_options, providers=PROVIDERS)
FP32_DECODER_SESSION = ort.InferenceSession(str(FP32_DECODER), sess_options=session_options, providers=PROVIDERS)

def decoder_inputs(image_id, memory):
    ids = np.zeros((3, 31), dtype=np.int64)
    mask = np.zeros((3, 31), dtype=np.int64)
    for index, tokens in enumerate(caption_rows[image_id][:3]):
        prefix = tokens[:-1][:31]
        ids[index, :len(prefix)] = np.asarray(prefix, dtype=np.int64)
        mask[index, :len(prefix)] = 1
    return {
        "decoder_input_ids": ids,
        "visual_memory": np.repeat(memory, 3, axis=0).astype(np.float32, copy=False),
        "decoder_attention_mask": mask,
    }

class EncoderCalibrationReader(CalibrationDataReader):
    def __init__(self, rows):
        self.rows = list(rows)
        self.rewind()
    def rewind(self):
        self.iterator = iter(self.rows)
    def get_next(self):
        try:
            row = next(self.iterator)
        except StopIteration:
            return None
        return {"image": preprocess_image(all_image_matches[row["image_id"]])}

class DecoderCalibrationReader(CalibrationDataReader):
    def __init__(self, rows):
        self.rows = list(rows)
        self.rewind()
    def rewind(self):
        self.iterator = iter(self.rows)
    def get_next(self):
        try:
            row = next(self.iterator)
        except StopIteration:
            return None
        image = preprocess_image(all_image_matches[row["image_id"]])
        memory = FP32_ENCODER_SESSION.run(None, {"image": image})[0]
        return decoder_inputs(row["image_id"], memory)

# Smoke test toàn bộ contract trước khi quantize.
sample_row = calibration_rows[0]
sample_image = preprocess_image(all_image_matches[sample_row["image_id"]])
sample_memory = FP32_ENCODER_SESSION.run(None, {"image": sample_image})[0]
sample_decoder_inputs = decoder_inputs(sample_row["image_id"], sample_memory)
sample_logits = FP32_DECODER_SESSION.run(None, sample_decoder_inputs)[0]
assert sample_memory.shape == (1, 49, 256)
assert sample_logits.shape == (3, 31, 10080)
assert np.isfinite(sample_memory).all() and np.isfinite(sample_logits).all()

write_json(AUDITS / "cell03_calibration_readers.json", {
    "schema_version": "dense_mapped_logit_kd_calibration_readers_v1",
    "created_at_utc": now(),
    "status": "passed",
    "encoder_input_shape": list(sample_image.shape),
    "memory_shape": list(sample_memory.shape),
    "decoder_logits_shape": list(sample_logits.shape),
    "calibration_images": CALIBRATION_COUNT,
    "captions_per_decoder_calibration_sample": 3,
    "test_set_used": False,
})

print("Preprocessing contract   : PASSED")
print("Encoder reader           : READY")
print("Decoder reader           : READY")
print("FP32 runtime smoke       : PASSED")


Preprocessing contract   : PASSED
Encoder reader           : READY
Decoder reader           : READY
FP32 runtime smoke       : PASSED


## Cell 4 — Tạo static QDQ và dynamic-weight INT8 candidates


In [4]:
STATIC_DIR = CANDIDATES / "static_qdq_int8"
DYNAMIC_DIR = CANDIDATES / "dynamic_weight_int8"
STATIC_DIR.mkdir(parents=True, exist_ok=True)
DYNAMIC_DIR.mkdir(parents=True, exist_ok=True)

# ORT quantizer tạo file *-inferred.onnx cạnh model nguồn. Kaggle mount
# notebook inputs read-only, vì vậy phải quantize từ bản sao writable.
QUANTIZER_SCRATCH = OUT / "_quantizer_scratch"
if QUANTIZER_SCRATCH.exists():
    shutil.rmtree(QUANTIZER_SCRATCH)
QUANTIZER_SCRATCH.mkdir(parents=True, exist_ok=True)
QUANTIZER_ENCODER_SOURCE = QUANTIZER_SCRATCH / "encoder_fp32.onnx"
QUANTIZER_DECODER_SOURCE = QUANTIZER_SCRATCH / "decoder_fp32.onnx"
shutil.copy2(FP32_ENCODER, QUANTIZER_ENCODER_SOURCE)
shutil.copy2(FP32_DECODER, QUANTIZER_DECODER_SOURCE)
assert QUANTIZER_ENCODER_SOURCE.is_file() and QUANTIZER_DECODER_SOURCE.is_file()

candidate_registry = []

def record_candidate(name, directory, status, error=None):
    encoder_path = directory / "encoder_int8.onnx"
    decoder_path = directory / "decoder_int8.onnx"
    row = {
        "name": name,
        "status": status,
        "error": error,
        "encoder": str(encoder_path),
        "decoder": str(decoder_path),
    }
    if status == "created":
        for path in (encoder_path, decoder_path):
            assert path.is_file() and path.stat().st_size > 1_000_000, path
            onnx.checker.check_model(onnx.load(str(path), load_external_data=True), full_check=True)
        row.update({
            "encoder_bytes": encoder_path.stat().st_size,
            "decoder_bytes": decoder_path.stat().st_size,
            "total_bytes": encoder_path.stat().st_size + decoder_path.stat().st_size,
            "encoder_sha256": sha256(encoder_path),
            "decoder_sha256": sha256(decoder_path),
        })
    candidate_registry.append(row)

started = time.perf_counter()
try:
    quantize_static(
        model_input=str(QUANTIZER_ENCODER_SOURCE),
        model_output=str(STATIC_DIR / "encoder_int8.onnx"),
        calibration_data_reader=EncoderCalibrationReader(calibration_rows),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
        calibrate_method=CalibrationMethod.MinMax,
        op_types_to_quantize=["Conv", "MatMul", "Gemm"],
        extra_options={"ActivationSymmetric": False, "WeightSymmetric": True},
    )
    quantize_static(
        model_input=str(QUANTIZER_DECODER_SOURCE),
        model_output=str(STATIC_DIR / "decoder_int8.onnx"),
        calibration_data_reader=DecoderCalibrationReader(calibration_rows),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
        calibrate_method=CalibrationMethod.MinMax,
        op_types_to_quantize=["MatMul", "Gemm"],
        extra_options={"ActivationSymmetric": False, "WeightSymmetric": True},
    )
    record_candidate("static_qdq_int8", STATIC_DIR, "created")
except Exception as exc:
    record_candidate("static_qdq_int8", STATIC_DIR, "failed", f"{type(exc).__name__}: {exc}")

try:
    quantize_dynamic(
        model_input=str(QUANTIZER_ENCODER_SOURCE),
        model_output=str(DYNAMIC_DIR / "encoder_int8.onnx"),
        per_channel=True,
        reduce_range=False,
        weight_type=QuantType.QInt8,
        op_types_to_quantize=["MatMul", "Gemm"],
    )
    quantize_dynamic(
        model_input=str(QUANTIZER_DECODER_SOURCE),
        model_output=str(DYNAMIC_DIR / "decoder_int8.onnx"),
        per_channel=True,
        reduce_range=False,
        weight_type=QuantType.QInt8,
        op_types_to_quantize=["MatMul", "Gemm"],
    )
    record_candidate("dynamic_weight_int8", DYNAMIC_DIR, "created")
except Exception as exc:
    record_candidate("dynamic_weight_int8", DYNAMIC_DIR, "failed", f"{type(exc).__name__}: {exc}")

shutil.rmtree(QUANTIZER_SCRATCH)
assert any(row["status"] == "created" for row in candidate_registry), candidate_registry
write_json(RESULTS / "int8_candidate_registry.json", {
    "schema_version": "dense_mapped_logit_kd_int8_candidate_registry_v1",
    "created_at_utc": now(),
    "candidates": candidate_registry,
    "quantization_seconds": time.perf_counter() - started,
    "calibration_split": "Karpathy Validation",
    "calibration_images": CALIBRATION_COUNT,
    "test_set_used": False,
})

for row in candidate_registry:
    print(f"{row['name']:<24}: {row['status'].upper()}")
print("Calibration images      :", CALIBRATION_COUNT)
print("Held-out Test           : SEALED / NOT USED")


static_qdq_int8         : CREATED
dynamic_weight_int8     : CREATED
Calibration images      : 128
Held-out Test           : SEALED / NOT USED


## Cell 5 — Disjoint Validation parity, latency và candidate selection


In [5]:
def cosine_similarity(left, right):
    left = np.asarray(left, dtype=np.float64).reshape(-1)
    right = np.asarray(right, dtype=np.float64).reshape(-1)
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    assert denominator > 0
    return float(np.dot(left, right) / denominator)

def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=np.float64), q))

FP32_TOTAL_BYTES = FP32_ENCODER.stat().st_size + FP32_DECODER.stat().st_size
GATES = {
    "encoder_cosine_mean_minimum": 0.99,
    "decoder_logits_cosine_mean_minimum": 0.99,
    "valid_position_top1_agreement_minimum": 0.95,
    "package_reduction_required": True,
    "finite_and_reload_required": True,
}

evaluated = []
for candidate in candidate_registry:
    if candidate["status"] != "created":
        continue
    encoder_session = ort.InferenceSession(candidate["encoder"], sess_options=session_options, providers=PROVIDERS)
    decoder_session = ort.InferenceSession(candidate["decoder"], sess_options=session_options, providers=PROVIDERS)
    encoder_cosines = []
    decoder_cosines = []
    top1_agreements = []
    encoder_ms = []
    decoder_ms = []
    finite = True
    per_image = []
    for row in parity_rows:
        image = preprocess_image(all_image_matches[row["image_id"]])
        start = time.perf_counter()
        fp32_memory = FP32_ENCODER_SESSION.run(None, {"image": image})[0]
        fp32_encoder_ms = (time.perf_counter() - start) * 1000
        start = time.perf_counter()
        int8_memory = encoder_session.run(None, {"image": image})[0]
        encoder_ms.append((time.perf_counter() - start) * 1000)

        fp32_inputs = decoder_inputs(row["image_id"], fp32_memory)
        int8_inputs = decoder_inputs(row["image_id"], int8_memory)
        start = time.perf_counter()
        fp32_logits = FP32_DECODER_SESSION.run(None, fp32_inputs)[0]
        fp32_decoder_ms = (time.perf_counter() - start) * 1000
        start = time.perf_counter()
        int8_logits = decoder_session.run(None, int8_inputs)[0]
        decoder_ms.append((time.perf_counter() - start) * 1000)

        lengths = int8_inputs["decoder_attention_mask"].sum(axis=1).astype(int)
        agreements = []
        for beam, length in enumerate(lengths):
            position = max(0, length - 1)
            agreements.append(
                int(np.argmax(fp32_logits[beam, position])) == int(np.argmax(int8_logits[beam, position]))
            )
        memory_cosine = cosine_similarity(fp32_memory, int8_memory)
        logits_cosine = cosine_similarity(fp32_logits, int8_logits)
        agreement = float(np.mean(agreements))
        encoder_cosines.append(memory_cosine)
        decoder_cosines.append(logits_cosine)
        top1_agreements.append(agreement)
        finite = finite and bool(np.isfinite(int8_memory).all() and np.isfinite(int8_logits).all())
        per_image.append({
            "image_id": row["image_id"],
            "encoder_cosine": memory_cosine,
            "decoder_logits_cosine": logits_cosine,
            "valid_position_top1_agreement": agreement,
            "fp32_encoder_ms": fp32_encoder_ms,
            "int8_encoder_ms": encoder_ms[-1],
            "fp32_decoder_ms": fp32_decoder_ms,
            "int8_decoder_ms": decoder_ms[-1],
        })

    total_bytes = int(candidate["total_bytes"])
    reduction = 1.0 - total_bytes / FP32_TOTAL_BYTES
    observed = {
        "encoder_cosine_mean": statistics.fmean(encoder_cosines),
        "encoder_cosine_minimum": min(encoder_cosines),
        "decoder_logits_cosine_mean": statistics.fmean(decoder_cosines),
        "decoder_logits_cosine_minimum": min(decoder_cosines),
        "valid_position_top1_agreement_mean": statistics.fmean(top1_agreements),
        "valid_position_top1_agreement_minimum": min(top1_agreements),
        "encoder_p50_ms": percentile(encoder_ms, 50),
        "decoder_p50_ms": percentile(decoder_ms, 50),
        "candidate_bytes": total_bytes,
        "fp32_bytes": FP32_TOTAL_BYTES,
        "package_reduction_fraction": reduction,
        "finite_and_reload": finite,
    }
    gates = {
        "encoder_cosine": observed["encoder_cosine_mean"] >= GATES["encoder_cosine_mean_minimum"],
        "decoder_logits_cosine": observed["decoder_logits_cosine_mean"] >= GATES["decoder_logits_cosine_mean_minimum"],
        "valid_position_top1_agreement": observed["valid_position_top1_agreement_mean"] >= GATES["valid_position_top1_agreement_minimum"],
        "package_reduction": reduction > 0,
        "finite_and_reload": finite,
    }
    evaluated.append({
        **candidate,
        "parity_images": PARITY_COUNT,
        "observed": observed,
        "gates": gates,
        "eligible": all(gates.values()),
        "per_image": per_image,
    })

eligible = sorted(
    (row for row in evaluated if row["eligible"]),
    key=lambda row: (row["observed"]["candidate_bytes"], row["observed"]["encoder_p50_ms"] + row["observed"]["decoder_p50_ms"], row["name"]),
)
selected_candidate = eligible[0] if eligible else None
decision = "int8_candidate_selected" if selected_candidate else "retain_fp32_no_int8_candidate_passed"

technical_result = {
    "schema_version": "dense_mapped_logit_kd_int8_validation_preflight_result_v1",
    "created_at_utc": now(),
    "status": "passed",
    "decision": decision,
    "thresholds": GATES,
    "calibration_images": CALIBRATION_COUNT,
    "parity_holdout_images": PARITY_COUNT,
    "cohorts_disjoint": True,
    "evaluated": evaluated,
    "selected_candidate": selected_candidate,
    "quality_claim_made": False,
    "full_validation_quality_required": selected_candidate is not None,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(RESULTS / "int8_validation_preflight_result.json", technical_result)
write_json(AUDITS / "cell05_candidate_selection.json", {
    "status": "passed", "decision": decision,
    "eligible_candidates": [row["name"] for row in eligible],
    "selected_candidate": selected_candidate["name"] if selected_candidate else None,
    "test_set_used": False,
})

for row in evaluated:
    print(
        f"{row['name']:<24} | eligible={row['eligible']} | "
        f"enc_cos={row['observed']['encoder_cosine_mean']:.6f} | "
        f"dec_cos={row['observed']['decoder_logits_cosine_mean']:.6f} | "
        f"top1={row['observed']['valid_position_top1_agreement_mean']:.2%} | "
        f"size={row['observed']['package_reduction_fraction']:.2%}"
    )
print("Decision                 :", decision)
print("Selected candidate       :", selected_candidate["name"] if selected_candidate else "FP32 fallback")
print("Held-out Test            : SEALED / NOT USED")


static_qdq_int8          | eligible=False | enc_cos=0.516637 | dec_cos=0.976213 | top1=93.75% | size=56.57%
dynamic_weight_int8      | eligible=True | enc_cos=0.999767 | dec_cos=0.999818 | top1=100.00% | size=38.37%
Decision                 : int8_candidate_selected
Selected candidate       : dynamic_weight_int8
Held-out Test            : SEALED / NOT USED


## Cell 6 — Package, manifest và full Validation authorization


In [6]:
PACKAGE_FP32_ENCODER = PACKAGE / "encoder_fp32.onnx"
PACKAGE_FP32_DECODER = PACKAGE / "decoder_fp32.onnx"
shutil.copy2(FP32_ENCODER, PACKAGE_FP32_ENCODER)
shutil.copy2(FP32_DECODER, PACKAGE_FP32_DECODER)
shutil.copy2(VOCAB_PATH, PACKAGE / VOCAB_PATH.name)
shutil.copy2(DECODER_CONTRACT_PATH, PACKAGE / DECODER_CONTRACT_PATH.name)
shutil.copy2(DEPLOYMENT_CONTRACT, PACKAGE / "fp32_deployment_contract.json")

if selected_candidate is not None:
    PACKAGE_INT8_ENCODER = PACKAGE / "encoder_int8.onnx"
    PACKAGE_INT8_DECODER = PACKAGE / "decoder_int8.onnx"
    shutil.copy2(selected_candidate["encoder"], PACKAGE_INT8_ENCODER)
    shutil.copy2(selected_candidate["decoder"], PACKAGE_INT8_DECODER)
    authorization_next = {
        "schema_version": "dense_mapped_logit_kd_int8_full_validation_quality_authorized_v1",
        "created_at_utc": now(),
        "status": "authorized",
        "selected_candidate": selected_candidate["name"],
        "selected_seed": 123,
        "required_next": "paired FP32-vs-INT8 beam-3 quality evaluation on all 5000 Karpathy Validation images",
        "quality_gates": {
            "CIDEr-D_minimum_delta": -0.02,
            "BLEU-4_minimum_delta": -0.01,
            "CE_maximum_delta": 0.05,
            "finite_required": True,
        },
        "constraints": {
            "same_5000_validation_images": True,
            "same_beam_size": 3,
            "same_length_penalty": 0.7,
            "new_training": False,
            "checkpoint_reselection": False,
            "test_opening": False,
            "android_benchmark_before_quality_gate": False,
        },
        "artifacts": {
            "encoder_int8": {"relative_path": "package/encoder_int8.onnx", "sha256": sha256(PACKAGE_INT8_ENCODER)},
            "decoder_int8": {"relative_path": "package/decoder_int8.onnx", "sha256": sha256(PACKAGE_INT8_DECODER)},
            "encoder_fp32": {"relative_path": "package/encoder_fp32.onnx", "sha256": sha256(PACKAGE_FP32_ENCODER)},
            "decoder_fp32": {"relative_path": "package/decoder_fp32.onnx", "sha256": sha256(PACKAGE_FP32_DECODER)},
        },
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    NEXT_AUTH_PATH = OUT / "XLKD05_Dense_Mapped_LogitKD_INT8_FULL_VALIDATION_QUALITY_AUTHORIZED.json"
    write_json(NEXT_AUTH_PATH, authorization_next)
else:
    NEXT_AUTH_PATH = None

artifact_paths = []
for directory in (AUDITS, CONFIGS, CANDIDATES, RESULTS, PACKAGE):
    artifact_paths.extend(path for path in directory.rglob("*") if path.is_file())
if NEXT_AUTH_PATH is not None:
    artifact_paths.append(NEXT_AUTH_PATH)

manifest_path = MANIFESTS / "int8_validation_preflight_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["relative_path", "bytes", "sha256"])
    writer.writeheader()
    for path in sorted(set(artifact_paths), key=lambda item: item.relative_to(OUT).as_posix()):
        writer.writerow({
            "relative_path": path.relative_to(OUT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        })

with manifest_path.open("r", encoding="utf-8", newline="") as handle:
    manifest_rows = list(csv.DictReader(handle))
for row in manifest_rows:
    path = OUT / row["relative_path"]
    assert path.is_file() and path.stat().st_size == int(row["bytes"])
    assert sha256(path) == row["sha256"]

completion = {
    "schema_version": "dense_mapped_logit_kd_int8_validation_preflight_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": decision,
    "selected_seed": 123,
    "selected_candidate": selected_candidate["name"] if selected_candidate else None,
    "calibration_images": CALIBRATION_COUNT,
    "parity_holdout_images": PARITY_COUNT,
    "cohorts_disjoint": True,
    "full_validation_quality_authorized": selected_candidate is not None,
    "quality_claim_made": False,
    "physical_android_benchmark_performed": False,
    "manifest": {
        "relative_path": manifest_path.relative_to(OUT).as_posix(),
        "rows": len(manifest_rows),
        "sha256": sha256(manifest_path),
    },
    "new_training": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
COMPLETE_MARKER = OUT / "XLKD05_Dense_Mapped_LogitKD_INT8_VALIDATION_PREFLIGHT_COMPLETE.json"
write_json(COMPLETE_MARKER, completion)

print("INT8 Validation preflight:", "COMPLETE")
print("Decision                 :", decision)
print("Selected candidate       :", completion["selected_candidate"])
print("Full Validation quality  :", "AUTHORIZED" if completion["full_validation_quality_authorized"] else "NOT AUTHORIZED")
print("Quality claim            : NOT YET")
print("Android benchmark        : NOT YET")
print("Held-out Test            : SEALED / NOT USED")
print("Manifest                 :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Complete marker          :", COMPLETE_MARKER)


INT8 Validation preflight: COMPLETE
Decision                 : int8_candidate_selected
Selected candidate       : dynamic_weight_int8
Full Validation quality  : AUTHORIZED
Quality claim            : NOT YET
Android benchmark        : NOT YET
Held-out Test            : SEALED / NOT USED
Manifest                 : 20 / 20 VERIFIED
Complete marker          : /kaggle/working/XLKD05_Dense_Mapped_LogitKD_int8_validation_preflight/XLKD05_Dense_Mapped_LogitKD_INT8_VALIDATION_PREFLIGHT_COMPLETE.json
